# CARec with cached item embeddings

- Load one embedding condition and the saved interaction splits.
- Product text and sampling belong in [data_exploration.ipynb](data_exploration.ipynb). Encoder inference belongs in [carec_encoder_comparison.ipynb](carec_encoder_comparison.ipynb).


## What CARec already implements

- Build a user/product graph from training interactions and aggregate neighbours with LightGCN.
- Phase 1 learns the user embedding table while keeping cached product vectors fixed and bypassing the MLP.
- Phase 2 freezes the user table and learns an MLP that transforms the product vectors. Graph propagation still runs.
- Alignment brings interacting users and products closer; uniformity prevents collapse. The text encoder stays frozen.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "requirements.txt").is_file()
)
DATASET_ID = "electronics_carec_035_seed2023"
PREPARED_DIR = PROJECT_ROOT / "data/processed" / DATASET_ID
EMBEDDING_NAME = "all-MiniLM-L6-v2.npz"
EMBEDDING_PATH = PROJECT_ROOT / "artifacts/embeddings" / DATASET_ID / EMBEDDING_NAME


## Load item embeddings

- Keep each vector paired with its original product ID.
- Change `EMBEDDING_NAME` to select another cached condition.


In [2]:
with np.load(EMBEDDING_PATH, allow_pickle=False) as saved:
    embedding_item_ids = saved["item_ids"]
    item_embeddings = saved["embeddings"]
    embedding_model_id = str(saved["model_id"])
    embedding_prompt = str(saved["prompt"])

print(embedding_model_id)
print(f"Embeddings: {item_embeddings.shape}")
print(f"Prompt: {embedding_prompt!r}")


sentence-transformers/all-MiniLM-L6-v2
Embeddings: (34220, 384)
Prompt: ''


## Load interaction splits

- Keep user and product IDs as strings.
- Only `train` interactions may enter the graph. Validation, test, and cold interactions stay held out.


In [3]:
partitions = {}
for name in ("train", "valid", "test", "cold"):
    partitions[name] = pd.read_csv(
        PREPARED_DIR / f"{name}.tsv.gz",
        sep="\t",
        dtype={"user_id:token": str, "item_id:token": str},
        float_precision="round_trip",
    )
    print(f"{name}: {len(partitions[name]):,} interactions")

train = partitions["train"]
valid = partitions["valid"]
test = partitions["test"]
cold = partitions["cold"]


train: 452,606 interactions
valid: 14,853 interactions
test: 133,274 interactions
cold: 23,163 interactions


## Next small steps

- Connect the saved splits to RecBole and align vectors with its internal product IDs, including padding.
- Instantiate the original CARec model and inspect its training graph before adding training.
- Add the two training phases, then warm/cold evaluation and the content-only baseline.
- Use native MiniLM vectors for the first integration. Settle the shared dimension/PCA protocol before comparing encoders.
